In [ ]:
import json
import re
import pandas as pd


def categorize_error_text_level(gold_text, pred_text):
    """基于文本与单词重叠逻辑判断边界错误，摆脱对精准字符索引的依赖"""
    g = str(gold_text).strip().lower()
    p = str(pred_text).strip().lower()

    # 1. 完全一致
    if g == p:
        return "Correct"

    # 2. 预测为空/彻底没有预测
    if not p:
        return "Wrong Entity"

    # 提取纯字母数字的单词列表
    g_words = re.findall(r"\w+", g)
    p_words = re.findall(r"\w+", p)

    g_set = set(g_words)
    p_set = set(p_words)

    # 计算共同单词
    common = g_set.intersection(p_set)

    # 没有共同词，直接算错词
    if not common:
        return "Wrong Entity"

    # 3. 边界划长了 (Longer): 预测包含了真实答案的所有核心词，但词数更多
    if g_set.issubset(p_set) and len(p_words) > len(g_words):
        return "Boundary Issue (Longer)"

    # 4. 边界划短了 (Shorter): 预测的词全部来自真实答案，但漏掉了部分词
    if p_set.issubset(g_set) and len(p_words) < len(g_words):
        return "Boundary Issue (Shorter)"

    # 5. 其他情况（交叉、主体替换等）
    return "Wrong Entity"


def evaluate_table6_errors_v2(gold_json_path, pred_json_path):
    # 读取 JSONL 格式的测试集
    with open(gold_json_path, "r", encoding="utf-8") as f:
        ground_truth = [json.loads(line) for line in f if line.strip()]

    # 读取模型预测结果
    with open(pred_json_path, "r", encoding="utf-8") as f:
        predictions = json.load(f)

    error_counts = {
        "Total Errors": 0,
        "Boundary Issue (Longer)": 0,
        "Boundary Issue (Shorter)": 0,
        "Wrong Entity": 0,
    }

    for item in ground_truth:
        qid = item["id"]
        true_text = (
            item["answers"]["text"][0] if item.get("answers") else ""
        )  #[cite: 2]
        pred_text = predictions.get(qid, "")

        err_type = categorize_error_text_level(true_text, pred_text)

        if err_type != "Correct":
            error_counts["Total Errors"] += 1
            error_counts[err_type] += 1

    return error_counts


# ====================
# 🚀 运行脚本
# ====================
gold_file = "/content/drive/MyDrive/数据/Test/Data_test.json"
pred_file = "/content/drive/MyDrive/数据/100%_data_train_BiolinkBert_results/predict_predictions.json"

res = evaluate_table6_errors_v2(gold_file, pred_file)

df_table6 = pd.DataFrame([res])
print("=== Table 6. Error Distribution for ChatMed-VHI (GPT-4.1) ===")
print(df_table6.to_string(index=False))

=== Table 6. Error Distribution for ChatMed-VHI (GPT-4.1) ===
 Total Errors  Boundary Issue (Longer)  Boundary Issue (Shorter)  Wrong Entity
          134                       88                        18            28
